#### qLoRA Finetuning

In [ ]:
!pip install -q -U transformers peft bitsandbytes datasets trl accelerate evaluate rouge_score sacrebleu

In [ ]:
import math
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig


# 1. Configuration (T4 GPU Optimized)

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_ID = "yahma/alpaca-cleaned"

4-bit quantization configuration (QLoRA)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # T4 GPUs use float16 (not bfloat16)
    bnb_4bit_use_double_quant=True,
)

# 2. Load Model & Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm(

Prepare model for 4-bit training & attach LoRA adapters

In [ ]:
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


# 3. Load & Format Dataset (Using a small subset for fast Colab execution)

In [ ]:
dataset = load_dataset(DATASET_ID, split="train[:600]")
#dataset = dataset.train_test_split(test_size=0.1, seed=42)

dataset

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 600
})

In [ ]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 540
    })
    test: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 60
    })
})

In [ ]:
df = load_dataset("yahma/alpaca-cleaned")
df

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})

In [ ]:
import pandas as pd

# Access the 'train' split of the DatasetDict before converting to pandas
pand_df = df['train'].to_pandas()
pand_df.head()

,output,input,instruction
0,1. Eat a balanced and nutritious diet: Make su...,,Give three tips for staying healthy.
1,"The three primary colors are red, blue, and ye...",,What are the three primary colors?
2,An atom is the basic building block of all mat...,,Describe the structure of an atom.
3,There are several ways to reduce air pollution...,,How can we reduce air pollution?
4,I had to make a difficult decision when I was ...,,Pretend you are a project manager of a constru...


In [ ]:
import numpy as np

# Filter rows where the 'input' column is not NaN and is not an empty string
filtered_df = pand_df[pand_df['input'].notna() & (pand_df['input'].str.strip() != "")]
filtered_df

,output,input,instruction
6,The fraction 4/16 is equivalent to 1/4 because...,4/16,Explain why the following fraction is equivale...
8,There are two spelling errors in the sentence....,He finnished his meal and left the resturant,Evaluate this sentence for spelling and gramma...
13,Animals: Elephant\nPlants: Oak tree\nMinerals:...,"Oak tree, copper ore, elephant","Classify the following into animals, plants, a..."
16,```\n# This is a function that calculates the ...,def factorialize(num):\n factorial = 1\n for...,Refactor this code and add comments.
22,She will play the piano beautifully for hours ...,She played the piano beautifully for hours and...,"Based on the information provided, rewrite the..."
...,...,...,...
51755,Yes,Text: John went out for a walk with his dog Ro...,You will be given a piece of text about an eve...
51756,True,Text: Michael Jordan is an American former pro...,You will be given a paragraph of text with var...
51757,True,Text: A tree fell over in the wind and caused ...,You will be given a piece of text about an eve...
51758,Backwards,"Steps: ['She takes out her books', 'The teache...",I will give you a list of steps. You need to ...


In [ ]:
def format_instruction(sample):
    prompt = f"### Instruction:\n{sample['instruction']}\n"
    if sample.get("input"):
        prompt += f"### Input:\n{sample['input']}\n"
    prompt += f"### Response:\n{sample['output']}"
    return {"text": prompt}

sample = {"instruction": "Give three tips for staying healthy.",
          "input": "1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases. 2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week. 3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.",
          "output": " "}
format_instruction(sample)

{'text': '### Instruction:\nGive three tips for staying healthy.\n### Input:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases. 2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week. 3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.\n### Response:\n '}

In [ ]:
train_dataset = dataset["train"].map(format_instruction)
eval_dataset = dataset["test"].map(format_instruction)

In [ ]:
train_dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 540
})

In [ ]:
train_dataset.to_pandas()

,output,input,instruction,text
0,Algae is a diverse group of aquatic organisms ...,,"Give a definition of ""algae"".","### Instruction:\nGive a definition of ""algae""..."
1,One reason someone should not use laser hair r...,Laser Hair Removal,Find a sentence that can be used to explain wh...,### Instruction:\nFind a sentence that can be ...
2,I needed to make a brief visit to the store.,I had to go to the store real quick.,Rewrite the given sentence in a more formal tone.,### Instruction:\nRewrite the given sentence i...
3,One possible scenario where blockchain technol...,,Describe a scenario in which blockchain techno...,### Instruction:\nDescribe a scenario in which...
4,A great idea for a family photoshoot could be ...,,Suggest a good idea for a family photoshoot.,### Instruction:\nSuggest a good idea for a fa...
...,...,...,...,...
535,A straight line is described by the linear equ...,,Derive the mathematical equation for a straigh...,### Instruction:\nDerive the mathematical equa...
536,Vacillation refers to the inability to make a ...,,"Provide a definition of the word ""vacillation"".",### Instruction:\nProvide a definition of the ...
537,History was made today as millions of Pluto an...,,Assuming millions of people have reached Pluto...,### Instruction:\nAssuming millions of people ...
538,Economics is a social science that studies how...,,"Write a simple definition of the word ""economi...",### Instruction:\nWrite a simple definition of...


# 4. Configure SFTTrainer for T4 Memory Limits

In [ ]:
training_args = SFTConfig(
    output_dir="./qlora-t4-output",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,      # Effective batch size = 8
    optim="paged_adamw_8bit",           # Prevents OOM spikes on T4
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    fp16=True,                          # Mixed precision for T4
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="no",
    report_to="none"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
)

# 5. Train the Model

In [ ]:
print("Ensuring correct float precision for T4 GPU...")
# Cast any accidental bfloat16 parameters to float32 to prevent PyTorch GradScaler errors
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

print("Starting QLoRA Training...")
train_result = trainer.train()

Ensuring correct float precision for T4 GPU...
Starting QLoRA Training...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
20,1.284554,1.304224,1.395152,0.683499,28949.000000
40,1.255396,1.239509,1.302446,0.686888,57060.000000
60,1.160328,1.232895,1.301937,0.687097,85990.000000
68,1.160328,1.232632,1.301686,0.686923,96773.000000


# 6. Compute All Evaluation Metrics

In [ ]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print(f"Training Loss:   {train_result.training_loss:.4f}")
print(f"Validation Loss: {eval_loss:.4f}")
print(f"Perplexity:      {perplexity:.4f}")

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
1.160328,1.232632,68,1.301686,0.686923,96773.000000


Training Loss:   1.2633
Validation Loss: 1.2326
Perplexity:      3.4302


In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
1.160328,1.232632,68,1.301686,0.686923,96773.000000


{'eval_loss': 1.2326323986053467,
 'eval_entropy': 1.3016855160395304,
 'eval_mean_token_accuracy': 0.6869231363137563,
 'eval_num_tokens': 96773.0}

# 7. Compute Text Generation Metrics (ROUGE & BLEU)

In [ ]:
print("\n--- Generating Predictions for ROUGE & BLEU ---")
rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")


--- Generating Predictions for ROUGE & BLEU ---


# Evaluate on 20 validation samples to keep inference fast on T4

In [ ]:
test_samples = dataset["test"].select(range(20))
predictions = []
references = []

model.eval()
for sample in test_samples:
    prompt = f"### Instruction:\n{sample['instruction']}\n"
    if sample.get("input"):
        prompt += f"### Input:\n{sample['input']}\n"
    prompt += "### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    # Decode only the newly generated tokens
    generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    predictions.append(generated_text.strip())
    references.append(sample["output"].strip())

rouge_scores = rouge.compute(predictions=predictions, references=references)
bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in references])

print("\n--- Final Evaluation Summary ---")
print(f"Eval Loss:  {eval_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")
print(f"ROUGE-1:    {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2:    {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L:    {rouge_scores['rougeL']:.4f}")
print(f"BLEU Score: {bleu_score['score']:.4f}")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging


--- Final Evaluation Summary ---
Eval Loss:  1.2326
Perplexity: 3.4302
ROUGE-1:    0.3654
ROUGE-2:    0.1907
ROUGE-L:    0.2955
BLEU Score: 4.3971
